In [1]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import duckdb as ddb


In [ ]:

ROOT = Path.cwd().resolve().parents[1]
RESULTS_DIR = ROOT / "capacity_based_results/"

In [7]:
print(RESULTS_DIR)

/Users/biancachiusano/Desktop/uva/thesis/energy-aware-vm-consolidation/capacity_based_results


In [16]:
f_placements_mm_cpu = pd.read_parquet(RESULTS_DIR / "MM_BFD_CPU/failed_placements_MM_BFD_CPU.parquet")
f_placements_mm_power = pd.read_parquet(RESULTS_DIR / "MM_PBFD/failed_placements_MM_PBFD.parquet")

In [12]:
f_placements_mm_cpu.head()

,vm_id,source_node,vm_power,vm_cpu,vm_memory_mb,insufficient_cpu_hosts,insufficient_memory_hosts,insufficient_power_hosts
0,3b02e532,e4362c19,1.41,32.0,92000.0,0,28,0
1,0431772d,e4362c19,0.26,16.0,46000.0,0,28,0
2,0041e677,e4362c19,0.15,16.0,46000.0,0,28,0
3,404b13ee,e4362c19,0.18,8.0,23000.0,0,28,0
4,40d350bf,e4362c19,0.09,8.0,23000.0,0,28,0


In [18]:
def analyze_constraints(df):
    constraints = [
        "insufficient_memory_hosts",
        "insufficient_cpu_hosts",
        "insufficient_power_hosts",
    ]

    df["main_constraint"] = df[constraints].idxmax(axis=1)

    print("Percentage of failed placements:")
    print(df["main_constraint"].value_counts(normalize=True) * 100)

analyze_constraints(f_placements_mm_cpu)
analyze_constraints(f_placements_mm_power)

Percentage of failed placements:
main_constraint
insufficient_cpu_hosts       91.400651
insufficient_memory_hosts     8.599349
Name: proportion, dtype: float64
Percentage of failed placements:
main_constraint
insufficient_cpu_hosts       91.478938
insufficient_memory_hosts     8.521062
Name: proportion, dtype: float64


In [1]:
import duckdb as ddb 

con = ddb.connect(":memory:")

In [8]:
con = ddb.connect()

con.execute(f"""
    SELECT
        MIN(timestamp) AS min_timestamp,
        MAX(timestamp) AS max_timestamp
    FROM read_parquet('/Users/biancachiusano/Desktop/uva/thesis/energy-aware-vm-consolidation/capacity_based_results/MM_CPU_BFD_5_25/simulated_MM_CPU_BFD_5_25.parquet')
""").df()

,min_timestamp,max_timestamp
0,2025-01-01 00:00:00+01:00,2025-02-28 23:57:00+01:00


In [9]:
con.execute(f"""
    SELECT
        MIN(timestamp) AS min_timestamp,
        MAX(timestamp) AS max_timestamp
    FROM read_parquet('/Users/biancachiusano/Desktop/uva/thesis/energy-aware-vm-consolidation/capacity_based_results/MM_CPU_BFD_0_30/simulated_MM_CPU_BFD_0_30.parquet')
""").df()

,min_timestamp,max_timestamp
0,2024-12-14 01:00:00+01:00,2025-04-13 23:57:00+02:00


In [2]:
con.execute(f"""CREATE OR REPLACE VIEW baseline AS SELECT * FROM read_parquet('/Users/biancachiusano/Desktop/uva/thesis/energy-aware-vm-consolidation/capacity_based_results/MM_CPU_BFD_5_25/simulated_MM_CPU_BFD_5_25.parquet')""")

In [5]:
con.execute(f"""CREATE OR REPLACE VIEW baseline AS SELECT * FROM read_parquet('/Users/biancachiusano/Desktop/uva/thesis/energy-aware-vm-consolidation/capacity_based_results/MM_CPU_BFD_0_30/simulated_MM_CPU_BFD_0_30.parquet')""")

In [6]:
con.query("SELECT timestamp from baseline GROUP BY timestamp").show()

┌──────────────────────────┐
│        timestamp         │
│ timestamp with time zone │
├──────────────────────────┤
│ 2025-04-12 12:03:00+02   │
│ 2025-04-12 12:09:00+02   │
│ 2025-04-12 12:18:00+02   │
│ 2025-04-12 12:33:00+02   │
│ 2025-04-12 13:36:00+02   │
│ 2025-04-12 14:12:00+02   │
│ 2025-04-12 14:15:00+02   │
│ 2025-04-12 14:24:00+02   │
│ 2025-04-12 14:36:00+02   │
│ 2025-04-12 15:12:00+02   │
│           ·              │
│           ·              │
│           ·              │
│ 2025-04-12 08:06:00+02   │
│ 2025-04-12 08:51:00+02   │
│ 2025-04-12 09:12:00+02   │
│ 2025-04-12 09:18:00+02   │
│ 2025-04-12 10:03:00+02   │
│ 2025-04-12 10:33:00+02   │
│ 2025-04-12 10:36:00+02   │
│ 2025-04-12 11:30:00+02   │
│ 2025-04-12 11:42:00+02   │
│ 2025-04-12 11:48:00+02   │
└──────────────────────────┘
           ? rows         
   (>9999 rows, 20 shown)  

